# Benchmarking `pyensmallen` estimator classes for m-estimation


In [1]:
import numpy as np
import pandas as pd
import pyensmallen
import scipy.optimize
import cvxpy as cp
from scipy.special import expit
import time

import statsmodels.api as sm

In [2]:
import jax
import jax.numpy as jnp
import optax
import functools
jax.config.update("jax_enable_x64", True)

## Data Generation

In [3]:
np.random.seed(42)
n, k = 1_000_000, 20

# Linear Regression Data
X_linear = np.random.randn(n, k)
print(true_params_linear := np.random.rand(k))
y_linear = X_linear @ true_params_linear

[0.51639859 0.94598022 0.23380001 0.55162275 0.97811966 0.24254699
 0.64702478 0.70271041 0.26476461 0.77362184 0.7817448  0.36874977
 0.72697004 0.06518613 0.72705723 0.38967364 0.03826155 0.39386005
 0.0438693  0.72142769]


## Linear Regression

### pyensmallen estimator API


In [4]:
linear_model = pyensmallen.LinearRegression(fit_intercept=False)


In [5]:
start_time = time.time()
linear_model.fit(X_linear, y_linear)
end_time = time.time()
result_linear_ens = linear_model.coef_
ensmallen_linear_time = end_time - start_time
print(f"pyensmallen linear regression time: {ensmallen_linear_time:.6f} seconds")


pyensmallen linear regression time: 0.430421 seconds


### scipy

In [7]:
start_time = time.time()
result_linear_scipy = scipy.optimize.minimize(
    fun=lambda b: np.sum((X_linear @ b - y_linear) ** 2),
    x0=np.zeros(X_linear.shape[1]),
    jac=lambda b: 2 * X_linear.T @ (X_linear @ b - y_linear),
).x
end_time = time.time()
scipy_linear_time = end_time - start_time
print(f"scipy linear regression time: {scipy_linear_time:.6f} seconds")

scipy linear regression time: 5.783995 seconds


### cvxpy

In [8]:
start_time = time.time()
b_linear = cp.Variable(k)
cost_linear = cp.norm(X_linear @ b_linear - y_linear, p=2) ** 2 / n
prob_linear = cp.Problem(cp.Minimize(cost_linear))
prob_linear.solve(solver=cp.SCS)
end_time = time.time()
cvxpy_linear_time = end_time - start_time
print(f"cvxpy linear regression time: {cvxpy_linear_time:.6f} seconds")


cvxpy linear regression time: 12.445588 seconds


### jax + optax

In [10]:
start_time = time.time()
X_jnp, y_jnp = jnp.array(X_linear), jnp.array(y_linear)

def compute_loss(beta):
    y_pred = jnp.dot(X_jnp, beta)
    loss = jnp.mean((y_pred - y_jnp) ** 2)
    return loss

params = jnp.array(jnp.zeros(X_linear.shape[1]))
solver = optax.lbfgs()
opt_state = solver.init(params)
value_and_grad = optax.value_and_grad_from_state(compute_loss)

# Optimization loop
for i in range(10):
    value, grad = value_and_grad(params, state=opt_state)
    updates, opt_state = solver.update(
        grad, opt_state, params, value=value, grad=grad, value_fn=compute_loss
    )
    params = optax.apply_updates(params, updates)
end_time = time.time()
jax_linear_time = end_time - start_time
print(f"jax linear regression time: {jax_linear_time:.6f} seconds")


jax linear regression time: 10.643983 seconds


### closed form

In [11]:
start_time = time.time()
np_lstsq_sol = np.linalg.lstsq(X_linear, y_linear, rcond=None)[0]
np_lstsq_time = end_time - start_time
print(f"numpy lstsq linear regression time: {np_lstsq_time:.6f} seconds")

numpy lstsq linear regression time: -3.416814 seconds


In [12]:
start_time = time.time()
sm_lstsq_sol = sm.OLS(y_linear, X_linear).fit().params
end_time = time.time()
sm_lstsq_time = end_time - start_time
print(f"statsmodels lstsq linear regression time: {sm_lstsq_time:.6f} seconds")


statsmodels lstsq linear regression time: 1.639195 seconds


## Comparison

In [13]:
lin_results = {
    "true": true_params_linear,
    "ensmallen": result_linear_ens,
    "scipy": result_linear_scipy,
    "cvxpy": b_linear.value,
    "jax": params,
    "np_lstsq": np_lstsq_sol,
    "sm_lstsq": sm_lstsq_sol,
}
lin_times = {
    "ensmallen": ensmallen_linear_time,
    "scipy": scipy_linear_time,
    "cvxpy": cvxpy_linear_time,
    "jax": jax_linear_time,
    "np_lstsq": np_lstsq_time,
    "sm_lstsq": sm_lstsq_time,
}
lin_df = pd.DataFrame(lin_results).T
lin_df["time"] = lin_df.index.map(lin_times)
lin_df

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,time
true,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,NaN
ensmallen,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,0.430421
scipy,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,5.783995
cvxpy,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,12.445588
jax,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,10.643983
np_lstsq,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,-3.416814
sm_lstsq,0.516399,0.94598,0.2338,0.551623,0.97812,0.242547,0.647025,0.70271,0.264765,0.773622,...,0.36875,0.72697,0.065186,0.727057,0.389674,0.038262,0.39386,0.043869,0.721428,1.639195


Numpy is obviously fastest here since we're doing a closed form solution. `pyensmallen` is fastest among MSE minimisers (and also beats statsmodels, which adds a lot of overhead over numpy despite using closed form).

## Logistic Regression

In [14]:
# Logistic Regression Data
n, k = 100_000, 20
X_logistic = np.random.randn(n, k)
print(true_params_logistic := np.random.rand(k))
p = expit(X_logistic @ true_params_logistic)
y_logistic = np.random.binomial(1, p)

[0.12535815 0.14795472 0.65017488 0.14922644 0.0475355  0.33917659
 0.23786835 0.44456079 0.08403914 0.58142796 0.58069396 0.92963811
 0.37971646 0.76615664 0.97499895 0.41514807 0.72909192 0.49428436
 0.14184926 0.21549712]


### pyensmallen estimator API


In [15]:
logistic_model = pyensmallen.LogisticRegression(fit_intercept=False)

In [18]:
start_time = time.time()
X_logistic2 = np.ascontiguousarray(
    X_logistic
)  # Ensure C-contiguous array for better performance
y_logistic2 = y_logistic.ravel()

logistic_model.fit(X_logistic2, y_logistic2)
end_time = time.time()
result_logistic_ens = logistic_model.coef_
ensmallen_logistic_time = end_time - start_time
print(f"pyensmallen logistic regression time: {ensmallen_logistic_time:.6f} seconds")


pyensmallen logistic regression time: 0.114029 seconds


### scipy

In [19]:
start_time = time.time()
result_logistic_scipy = scipy.optimize.minimize(
    fun=lambda b: -np.sum(
        y_logistic * np.log(expit(X_logistic @ b))
        + (1 - y_logistic) * np.log(1 - expit(X_logistic @ b))
    ),
    x0= np.zeros(X_logistic.shape[1]),
    jac=lambda b: X_logistic.T @ (expit(X_logistic @ b) - y_logistic),
).x
end_time = time.time()
scipy_logistic_time = end_time - start_time
print(f"scipy logistic regression time: {scipy_logistic_time:.6f} seconds")

/tmp/ipykernel_25856/2548110570.py:5: RuntimeWarning: divide by zero encountered in log
  + (1 - y_logistic) * np.log(1 - expit(X_logistic @ b))
/tmp/ipykernel_25856/2548110570.py:5: RuntimeWarning: invalid value encountered in multiply
  + (1 - y_logistic) * np.log(1 - expit(X_logistic @ b))


scipy logistic regression time: 0.607523 seconds


### cvxpy

In [20]:
start_time = time.time()
b_logistic = cp.Variable(k)
log_likelihood = cp.sum(
    cp.multiply(y_logistic, X_logistic @ b_logistic)
    - cp.logistic(X_logistic @ b_logistic)
)
prob_logistic = cp.Problem(cp.Maximize(log_likelihood))
prob_logistic.solve(solver=cp.SCS)
end_time = time.time()
cvxpy_logistic_time = end_time - start_time
print(f"cvxpy logistic regression time: {cvxpy_logistic_time:.6f} seconds")

cvxpy logistic regression time: 12.032684 seconds


### statsmodels

does IRLS

In [21]:
start_time = time.time()
sm_logit_res = sm.Logit(y_logistic, X_logistic).fit().params
end_time = time.time()
sm_logistic_time = end_time - start_time
print(f"statsmodels logistic regression time: {sm_logistic_time:.6f} seconds")

Optimization terminated successfully.
         Current function value: 0.432804
         Iterations 7
statsmodels logistic regression time: 0.235942 seconds


In [22]:
start_time = time.time()
X_jnp, y_jnp = jnp.array(X_logistic), jnp.array(y_logistic)

def logistic_likelihood(beta):
    z = jnp.dot(X_jnp, beta)
    h = jax.scipy.special.expit(z)
    loss = -jnp.sum(y_jnp * jnp.log(h) + (1 - y_jnp) * jnp.log1p(-h))
    return loss

params = jnp.array(jnp.zeros(X_logistic.shape[1]))
solver = optax.lbfgs()
opt_state = solver.init(params)
value_and_grad = optax.value_and_grad_from_state(logistic_likelihood)

# Optimization loop
for i in range(10):
    value, grad = value_and_grad(params, state=opt_state)
    updates, opt_state = solver.update(
        grad, opt_state, params, value=value, grad=grad, value_fn=logistic_likelihood
    )
    params = optax.apply_updates(params, updates)

end_time = time.time()
jax_logistic_time = end_time - start_time
print(f"jax logistic regression time: {jax_logistic_time:.6f} seconds")

jax logistic regression time: 11.477670 seconds


## comparison

In [23]:
logit_df = pd.DataFrame(
    {
        "true": true_params_logistic,
        "ensmallen": result_logistic_ens,
        "scipy": result_logistic_scipy,
        "cvxpy": b_logistic.value,
        "jax": params,
        "sm_logit": sm_logit_res,
    }
)

logit_times = {
    "ensmallen": ensmallen_logistic_time,
    "scipy": scipy_logistic_time,
    "cvxpy": cvxpy_logistic_time,
    "jax": jax_logistic_time,
    "sm_logit": sm_logistic_time,
}

logit_df = pd.DataFrame(logit_df).T
logit_df["time"] = logit_df.index.map(logit_times)
logit_df

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,time
true,0.125358,0.147955,0.650175,0.149226,0.047536,0.339177,0.237868,0.444561,0.084039,0.581428,...,0.929638,0.379716,0.766157,0.974999,0.415148,0.729092,0.494284,0.141849,0.215497,NaN
ensmallen,0.120316,0.158912,0.651493,0.133951,0.043761,0.319691,0.231893,0.450126,0.081442,0.572644,...,0.938037,0.351225,0.771810,0.974496,0.414057,0.724789,0.499617,0.127896,0.226819,0.114029
scipy,0.120316,0.158912,0.651493,0.133951,0.043761,0.319691,0.231893,0.450126,0.081442,0.572644,...,0.938037,0.351225,0.771810,0.974496,0.414057,0.724789,0.499617,0.127896,0.226819,0.607523
cvxpy,0.120316,0.158911,0.651490,0.133950,0.043761,0.319690,0.231892,0.450124,0.081442,0.572641,...,0.938032,0.351224,0.771806,0.974491,0.414055,0.724785,0.499615,0.127896,0.226818,12.032684
jax,0.120316,0.158912,0.651493,0.133951,0.043761,0.319691,0.231893,0.450126,0.081442,0.572644,...,0.938037,0.351225,0.771810,0.974496,0.414057,0.724789,0.499617,0.127896,0.226819,11.477670
sm_logit,0.120316,0.158912,0.651493,0.133951,0.043761,0.319691,0.231893,0.450126,0.081442,0.572644,...,0.938037,0.351225,0.771810,0.974496,0.414057,0.724789,0.499617,0.127896,0.226819,0.235942


Ensmallen is fastest among all libraries.

## Poisson Regression

In [24]:
n, k = 100_000, 10
# Poisson Regression Data
X_poisson = np.random.randn(n, k)
print(true_params_poisson := np.random.rand(k))
lambda_ = np.exp(X_poisson @ true_params_poisson)
y_poisson = np.random.poisson(lambda_)

[0.77477259 0.3642854  0.5015039  0.51659442 0.499755   0.92081531
 0.08654286 0.39559362 0.34675488 0.29274375]


### pyensmallen estimator API


In [25]:
poisson_model = pyensmallen.PoissonRegression(fit_intercept=False)

In [26]:
start_time = time.time()
poisson_model.fit(X_poisson, y_poisson)
end_time = time.time()
result_poisson_ens = poisson_model.coef_
ensmallen_poisson_time = end_time - start_time
print(f"pyensmallen poisson regression time: {ensmallen_poisson_time:.6f} seconds")


pyensmallen poisson regression time: 0.069259 seconds


### scipy

In [27]:
start_time = time.time()
result_poisson_scipy = scipy.optimize.minimize(
    fun=lambda b: np.sum(np.exp(X_poisson @ b) - y_poisson * (X_poisson @ b)),
    x0=np.zeros(X_poisson.shape[1]),
    jac=lambda b: X_poisson.T @ (np.exp(X_poisson @ b) - y_poisson),
).x
end_time = time.time()
scipy_poisson_time = end_time - start_time
print(f"scipy poisson regression time: {scipy_poisson_time:.6f} seconds")

scipy poisson regression time: 0.096860 seconds


/tmp/ipykernel_25856/269416605.py:3: RuntimeWarning: overflow encountered in exp
  fun=lambda b: np.sum(np.exp(X_poisson @ b) - y_poisson * (X_poisson @ b)),
/tmp/ipykernel_25856/269416605.py:5: RuntimeWarning: overflow encountered in exp
  jac=lambda b: X_poisson.T @ (np.exp(X_poisson @ b) - y_poisson),
/tmp/ipykernel_25856/269416605.py:5: RuntimeWarning: invalid value encountered in matmul
  jac=lambda b: X_poisson.T @ (np.exp(X_poisson @ b) - y_poisson),


### cvxpy

fails

In [28]:
# start_time = time.time()
# b_poisson = cp.Variable(k)
# z = X_poisson @ b_poisson
# cost_poisson = cp.sum(cp.exp(z) - cp.multiply(y_poisson, z)) / n
# prob_poisson = cp.Problem(cp.Minimize(cost_poisson))
# prob_poisson.solve(solver=cp.SCS)

# end_time = time.time()
# cvxpy_poisson_time = end_time - start_time
# print(f"cvxpy poisson regression time: {cvxpy_poisson_time:.6f} seconds")

Interrupted after 5 mins.

### statsmodels

In [29]:
start_time = time.time()
sm_poisson_res = sm.Poisson(y_poisson, X_poisson).fit().params
end_time = time.time()
sm_poisson_time = end_time - start_time
print(f"sm poisson regression time: {sm_poisson_time:.6f} seconds")

Optimization terminated successfully.
         Current function value: 1.374451
         Iterations 24
sm poisson regression time: 0.364055 seconds


## jax
switch to adam since lbfgs performs poorly (!TODO why?)

In [30]:
%%time
start_time = time.time()
X_jnp, y_jnp = jnp.array(X_poisson), jnp.array(y_poisson)

def poisson_likelihood(beta):
    z = jnp.dot(X_jnp, beta)
    lambda_ = jnp.exp(z)
    loss = jnp.sum(lambda_ - y_jnp * z)
    return loss


solver = optax.adam(1e-2)
adam_params = jnp.array(jnp.zeros(X_poisson.shape[1]))
opt_state = solver.init(adam_params)

@jax.jit
def update_step(params, opt_state):
    loss, grads = jax.value_and_grad(poisson_likelihood)(params)
    updates, opt_state = solver.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


for i in range(1000):
    adam_params, opt_state, loss = update_step(adam_params, opt_state)

end_time = time.time()
jax_poisson_time = end_time - start_time
print(f"jax poisson regression time: {jax_poisson_time:.6f} seconds")

jax poisson regression time: 0.460846 seconds
CPU times: user 865 ms, sys: 31.4 ms, total: 896 ms
Wall time: 461 ms


In [31]:
poi_df = pd.DataFrame(
    {
        "true": true_params_poisson,
        "ensmallen": result_poisson_ens,
        "scipy": result_poisson_scipy,
        "sm_poisson": sm_poisson_res,
        # "cvxpy": b_poisson.value,
        "jax": adam_params,
    }
)


poi_times = {
    "ensmallen": ensmallen_poisson_time,
    "scipy": scipy_poisson_time,
    "jax": jax_poisson_time,
    # "cvxpy": cvxpy_poisson_time,
    "sm_poisson": sm_poisson_time,
}

poi_df = pd.DataFrame(poi_df).T
poi_df["time"] = poi_df.index.map(poi_times)
poi_df

,0,1,2,3,4,5,6,7,8,9,time
true,0.774773,0.364285,0.501504,0.516594,0.499755,0.920815,0.086543,0.395594,0.346755,0.292744,NaN
ensmallen,0.773938,0.362786,0.502285,0.516460,0.499565,0.918613,0.085895,0.400557,0.343659,0.292875,0.069259
scipy,0.478017,0.221385,0.310777,0.313807,0.308946,0.559228,0.039160,0.248223,0.208351,0.181076,0.096860
sm_poisson,0.773938,0.362786,0.502285,0.516460,0.499565,0.918613,0.085895,0.400557,0.343659,0.292875,0.364055
jax,0.773938,0.362786,0.502285,0.516460,0.499565,0.918613,0.085895,0.400557,0.343659,0.292875,0.460846


Ensmallen is fastest again. CVXPY fails to converge after 5 minutes, as does statsmodels.